# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guided step-by-step template for loading and exploring a dataset using the `mlcroissant` library, referencing **all entities by their `@id` attributes** for clarity and reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

> **Note:** This dataset describes protein abundance, modifications, and sequences derived from human (Homo sapiens) samples analyzed via mass spectrometry.

In [ ]:
# Ensure `mlcroissant` is installed in the environment
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load Dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata (as a Python object, not a dict-like)
meta = dataset.metadata

print(f"Dataset Name: {meta.name}")
print(f"Description: {meta.description}\n")
print(f"Version: {meta.version}")
print(f"Published: {meta.datePublished}")
print(f"License: {meta.license}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values used for further data access and referencing.

> In Croissant, each entity (record set, field, column) has a unique `@id`. We will enumerate all record sets, their fields, and columns by their `@id`.

In [ ]:
ds = dataset # using the variable as in the template

print("Available record sets in the dataset:")
record_sets = list(ds.record_sets)
for rs in record_sets:
    print(f"- RecordSet: @id='{rs.id}', name='{rs.name}'")

# For each record set, show its fields and columns by @id
for rs in record_sets:
    print(f"\nFields for RecordSet @id='{rs.id}':")
    for fld in rs.fields:
        print(f"    - Field @id='{fld.id}', name='{fld.name}', dataType='{fld.dataType}'")

    print(f"\nColumns for RecordSet @id='{rs.id}':")
    for col in rs.columns:
        print(f"    - Column @id='{col.id}', name='{col.name}', source='{col.source}'")

## 3. Data Extraction
Load data from a specific record set (identified by its `@id`) into a DataFrame for analysis.

> The next cell loads **all** record sets into DataFrames for exploration. You can select the ones you wish to analyze further by their `@id` values.

In [ ]:
# List all recordset @id values (from the overview)
record_set_ids = [rs.id for rs in ds.record_sets]
print(f"Record sets found: {record_set_ids}\n")

# Create a DataFrame for each record set by @id
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet @id='{record_set_id}' ...")
    recs = list(ds.records(record_set=record_set_id))
    if recs:
        df = pd.DataFrame(recs)
        dataframes[record_set_id] = df
        print(f"  - Loaded DataFrame with shape: {df.shape}")
    else:
        print("  - No records found.")

# Preview the columns in the first record set with data
if dataframes:
    first_rs_id = next(iter(dataframes))
    print(f"\nColumns in RecordSet @id='{first_rs_id}': {dataframes[first_rs_id].columns.tolist()}")
    display(dataframes[first_rs_id].head())
else:
    print("No dataframes loaded. Please check data access or schema.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

In this example, we choose one record set and a numeric field (referenced by its column `@id`) for basic filtering and normalization.

In [ ]:
### ------- USER SELECTION REQUIRED BELOW --------
# You should change these to actual @id strings from your specific dataset overview
# Example selection based on the dataset's structure:
# Select a record set @id and a field/column @id

# If record sets and field ids are printed from previous cells, pick a 'main' record set (e.g. for protein data)
# Here, we will select automatically the first DataFrame for demonstration; update this for specific analysis
if not dataframes:
    raise ValueError("No dataframes are available.")

record_set_id = list(dataframes.keys())[0]
df = dataframes[record_set_id]  # The main protein table, for example
print(f"Working on record set: {record_set_id}")

# Choose a numeric field (column) by @id -- replace as needed, e.g. '@id': 'column:abundance' or similar
numeric_candidate = None
for c in df.columns:
    # Heuristically choose the first float/integer column, or pick by known @id
    if df[c].dtype in [int, float, 'int64', 'float64']:
        numeric_candidate = c
        break
if not numeric_candidate:
    # Try conversion
    for c in df.columns:
        try:
            df[c] = pd.to_numeric(df[c], errors='raise')
            numeric_candidate = c
            break
        except:
            pass
if not numeric_candidate:
    print("No suitable numeric field found for EDA!")
else:
    print(f"Using numeric field/column @id: {numeric_candidate}")

    # Example threshold
    threshold = df[numeric_candidate].mean()

    filtered_df = df[df[numeric_candidate] > threshold].copy()
    print(f"Filtered rows where {numeric_candidate} > mean ({threshold:.2f}): {filtered_df.shape[0]} records.")

    # Normalization (z-score)
    filtered_df[f"{numeric_candidate}_normalized"] = (filtered_df[numeric_candidate] - filtered_df[numeric_candidate].mean()) / filtered_df[numeric_candidate].std()

    print(filtered_df[[numeric_candidate, f"{numeric_candidate}_normalized"]].head())

    # Group by another field if applicable
    # Pick a categorical field with at least 2 unique values and not all unique (otherwise, groupby not meaningful)
    group_field = None
    for col in df.columns:
        if col == numeric_candidate:
            continue
        n_unique = df[col].nunique()
        if 1 < n_unique < 25 and df[col].dtype == object:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_candidate].mean()
        print(f"\nGrouped mean {numeric_candidate} by {group_field}:")
        print(grouped_df.head())
    else:
        print("No suitable grouping field found in this record set.")

## 5. Visualization
Visualize data distributions or relationships between fields. Reference your fields by their `@id` as used in the DataFrame.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Simple distribution plot for the numeric field
if numeric_candidate is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_candidate], kde=True)
    plt.xlabel(f"{numeric_candidate} (by @id)")
    plt.title(f"Distribution of {numeric_candidate}")
    plt.show()

    # Boxplot by group, if group_field available
    if group_field is not None:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_candidate, data=df)
        plt.title(f"{numeric_candidate} by {group_field}")
        plt.xlabel(f"{group_field} (by @id)")
        plt.ylabel(numeric_candidate)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, you have:
- Loaded metadata and tabular data from the [FAIR^2 Mass Spectrometry Extracellular Vesicle dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library
- Explored record sets, fields, and their `@id` values for precise referencing
- Extracted data into pandas DataFrames and performed basic filtering, normalization, and grouping
- Visualized distributions and (optionally) group differences for selected fields

To perform further domain-specific analyses, reference available field and record set `@id`s exposed during the overview, and apply additional analytical or visualization routines as appropriate for your research!
